## Apache Hadoop and Apache Spark

**Hadoop** is a broader ecosystem for distributed storage and batch processing. Its original processing engine, **MapReduce**, writes intermediate results to disk between every step. This makes it reliable for very large batch jobs, but slow for iterative workloads [1].

**Apache Spark** (and its Python API, **PySpark**) was built as a faster alternative to MapReduce. Spark keeps intermediate data **in memory** whenever possible, uses **lazy evaluation** (it builds a logical plan and only executes when an action like `.show()` or `.count()` is called), and optimizes that plan with a query optimizer (Catalyst) before running it. Spark can still read from and write to Hadoop's storage layer (HDFS - Haddop Distributed File System), and can run *on top of* Hadoop's cluster manager (YARN) — so Spark isn't a replacement for all of Hadoop, it's typically a replacement for the MapReduce processing engine, often reusing Hadoop's storage and resource management.

| | Hadoop MapReduce | Spark / PySpark |
|---|---|---|
| Processing model | Disk-based, step-by-step | In-memory (where possible), DAG-based |
| Evaluation | Eager | Lazy (builds a plan, executes on action) |
| Speed (iterative jobs) | Slower | Typically much faster |
| Storage | HDFS | Can use HDFS, S3, local files, etc. |
| APIs | Java-centric | Python (PySpark), Scala, Java, R, SQL |

## What does `SparkSession.builder`  control?

When you write:
```python
spark = SparkSession.builder \
    .appName("TaxiTipAnalysis") \
    .master("local[*]") \
    .getOrCreate()
```
the `.master(...)` argument tells Spark **where and how** to run its executors. The main options are:

- **`local`** — run Spark entirely on a single core of the current machine. No real parallelism, mostly for quick testing.
- **`local[N]`** — run Spark on the current machine using `N` cores as separate worker threads/processes. This is **simulated parallelism on one computer**: multiple cores of the *same* machine each process a partition of the data, but there's no network/cluster involved.
- **`local[*]`** — same as above, but automatically use **all available cores** on the current machine (this is what this notebook uses, and why `defaultParallelism` reports the number of local CPU cores).
- **`spark://host:port`** — connect to a **real standalone Spark cluster**, distributing work across multiple physical/virtual machines. This is **true distributed, multi-machine parallelism**.
- **`yarn`** — run on a Hadoop YARN-managed cluster, letting YARN allocate executors across the cluster's nodes.
- **`k8s://host:port`** — run on a Kubernetes cluster, with Spark executors launched as pods.
- **`mesos://host:port`** — run on an Apache Mesos-managed cluster (legacy option, less common today).

In this notebook `local[*]` is used. It allows for  *multi-core parallelism on one machine* — a good way to learn and prototype Spark's distributed programming model without needing an actual cluster. Swapping only the `.master(...)` string (e.g., to `yarn` or `spark://...`) lets the exact same PySpark code scale out to a real multi-node cluster with no other changes.

References

[1] https://hadoop.apache.org/docs/stable/index.html

[2] https://spark.apache.org/docs/latest/api/java/index.html

[3] https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.html

In [11]:
!pip install pyspark

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, avg

# Create a local Spark cluster simulation
spark = SparkSession.builder \
    .appName("TaxiTipAnalysis") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Session initialized. Version: {spark.version}")

Spark Session initialized. Version: 4.2.0


In [13]:
# Extract the configuration directly from the active Spark Context
#check how many coress PySpark uses
total_cores = spark.sparkContext.defaultParallelism

print(f"🧩 PySpark Architecture Status:")
print(f" - Execution Mode: Local Simulation")
print(f" - Active Distributed Cores Allocated: {total_cores}")
print(f" - Behind the scenes, Spark has divided Taxi dataset into at least {total_cores} parallel partitions!")


🧩 PySpark Architecture Status:
 - Execution Mode: Local Simulation
 - Active Distributed Cores Allocated: 8
 - Behind the scenes, Spark has divided Taxi dataset into at least 8 parallel partitions!


In [ ]:
file_path = "./taxi_data_subset.csv"

# Load the local CSV into a distributed Spark DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Verify the upload by printing the row count and schema
print(f"Successfully loaded custom dataset with {df.count():,} rows.")
df.printSchema()


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/taxi_data_subset.csv. SQLSTATE: 42K03

## Why use a PySpark DataFrame instead of pandas DataFrame?

The `groupBy("pickup_hour").agg(avg("tip_amount"))` operation below could also be written in pandas as something like `df.groupby(df['tpep_pickup_datetime'].dt.hour)['tip_amount'].mean()`. For a 50,000-row CSV like this one, pandas would actually run just fine — the real benefits of the PySpark version only show up as data and workload grow:

- **Scales beyond a single machine's memory.** pandas loads the *entire* dataset into RAM on one process. Spark partitions the data (we saw it split into partitions across the available cores above) and can spill to disk or spread across a whole cluster, so the same code works whether the file is 50,000 rows or 50 billion rows.
- **Parallel execution across cores/nodes.** pandas' `groupby` runs on a single CPU core. Spark's `groupBy`/`agg` here is executed as a distributed hash aggregation (visible in the `explain()` plan below as `HashAggregate` + `Exchange` steps), splitting the work across every core `local[*]` found — and across every machine in a cluster if `.master()` pointed to one.
- **Lazy evaluation and query optimization.** pandas executes each line immediately. Spark builds a logical plan first and lets Catalyst optimize it (e.g., pushing down only the two needed columns, `tpep_pickup_datetime` and `tip_amount`, in the `FileScan` step) before any computation happens — often avoiding unnecessary work entirely.
- **Same code, different scale.** Because the `.master(...)` setting is the only cluster-specific piece, this exact `groupBy`/`agg` code can move from a laptop to a multi-node cluster unchanged. A pandas script has no equivalent scaling path — it would need to be rewritten (e.g., in Dask or Spark) once the data outgrows one machine's memory.

The trade-off: for genuinely small data, PySpark's overhead (starting a Spark session, planning, task scheduling) can make it *slower* than pandas. PySpark's advantage grows with data size and cluster size, not on small datasets like this demo one.

## How to handle big data with pandas alone using `chunksize` and generators.

If PySpark isn't available, pandas offers a partial workaround: **`pd.read_csv(path, chunksize=N)`**. Instead of returning a DataFrame, this returns a `TextFileReader` — an **iterator/generator** that yields one DataFrame "chunk" of `N` rows at a time, reading only that chunk into memory before moving to the next.

```python
chunk_avg = {}   # hour -> [sum_of_tips, count]

for chunk in pd.read_csv("taxi_data_subset.csv", chunksize=10_000, parse_dates=["tpep_pickup_datetime"]):
    chunk["pickup_hour"] = chunk["tpep_pickup_datetime"].dt.hour
    grouped = chunk.groupby("pickup_hour")["tip_amount"].agg(["sum", "count"])
    for hour, row in grouped.iterrows():
        s, c = chunk_avg.get(hour, [0, 0])
        chunk_avg[hour] = [s + row["sum"], c + row["count"]]

hourly_avg = {h: round(s / c, 2) for h, (s, c) in chunk_avg.items()}
```

### compare pandas/generators/PySpark

| | pandas (full load) | pandas (`chunksize` / generator) | PySpark |
|---|---|---|---|
| Memory usage | Whole file in RAM | One chunk in RAM at a time | Partitioned; spills/distributes as needed |
| Can exceed machine's RAM? | No | Yes (file can be bigger than RAM) | Yes (and bigger than *cluster* RAM, via disk/shuffle) |
| Parallelism | Single core | **Still single core** — chunks are processed one after another, sequentially | True parallel execution across cores/nodes |
| Aggregation logic | Built-in `groupby` | **You write it manually** — combining partial sums/counts per chunk yourself | Built-in `groupBy`/`agg`, distributed automatically |
| Query optimization | None | None | Catalyst optimizer (column pruning, predicate pushdown, plan optimization) |
| Code changes to scale further | Rewrite needed | Rewrite needed (e.g. multiprocessing chunks, or move to Dask/Spark) | Same code scales from `local[*]` to a real cluster by changing `.master(...)` |

**The key distinction:** `chunksize` solves the *memory* problem (you're never holding more than one chunk at once) but not the *speed* problem — chunks are still read and processed one at a time on a single core, so it doesn't give you the parallel/distributed execution PySpark provides. It's a generator pattern for **out-of-core processing**, not for **parallel processing**. You could combine it with Python's `multiprocessing` to process chunks in parallel yourself, but at that point you're reimplementing (a much simpler version of) what Spark already does for you.

### Raw generator example.

This is generator written from scratch.  `chunksize` uses it internally.

```python
def read_in_batches(path, batch_size=10_000):
    with open(path) as f:
        header = f.readline()
        batch = []
        for line in f:
            batch.append(line)
            if len(batch) == batch_size:
                yield batch
                batch = []
        if batch:
            yield batch
```

This is more manual, as each line has to be parsed manually. It is useful to illustrate *why* `chunksize` exists.


In [ ]:
from pyspark.sql.functions import col, hour, avg, round

# 1. Extract the pickup hour from timestamp column
# 2. Group by hour, calculate average tip, and sort
hourly_tip_df = df.withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
                  .groupBy("pickup_hour") \
                  .agg(round(avg(col("tip_amount")), 2).alias("avg_tip_amount")) \
                  .orderBy("pickup_hour")

# Trigger the action to compute and display your results
hourly_tip_df.show(24)


+-----------+--------------+
|pickup_hour|avg_tip_amount|
+-----------+--------------+
|          0|          3.41|
|          1|          3.35|
|          2|          3.24|
|          3|          3.27|
|          4|          3.11|
|          5|           3.4|
|          6|          3.47|
|          7|           4.1|
|          8|          3.74|
|          9|          3.65|
|         10|          3.35|
|         11|          3.34|
|         12|          3.28|
|         13|          3.31|
|         14|          3.52|
|         15|          3.79|
|         16|          3.94|
|         17|          3.85|
|         23|          0.67|
+-----------+--------------+



In [ ]:

# Print the final execution plan to prove Spark is using distributed DAG processing
#Spark uses lazy evaluation. It means it does not execute code sequentially line-by-line.
#It waits until an action is triggered and works backward to build, optimize, and organize the steps into a logical roadmap.
#When  df.explain() is called, Spark reads from the bottom up (or inside out)
#to display how data flows from the storage disk into the final output.
hourly_tip_df.explain()

# Trigger action to compute and display results
hourly_tip_df.show(24)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [pickup_hour#62 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(pickup_hour#62 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=130]
      +- HashAggregate(keys=[pickup_hour#62], functions=[avg(tip_amount#30)])
         +- Exchange hashpartitioning(pickup_hour#62, 200), ENSURE_REQUIREMENTS, [plan_id=127]
            +- HashAggregate(keys=[pickup_hour#62], functions=[partial_avg(tip_amount#30)])
               +- Project [tip_amount#30, hour(tpep_pickup_datetime#18, Some(Etc/UTC)) AS pickup_hour#62]
                  +- FileScan csv [tpep_pickup_datetime#18,tip_amount#30] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/taxi_data_subset.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<tpep_pickup_datetime:timestamp,tip_amount:double>


+-----------+--------------+
|pickup_hour|avg_tip_amount|
+-----------+--------------+
|          0|          3